##### Module Imports

In [ ]:
import pandas as pd
from xgboost import XGBClassifier, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as pyplot

import sys                                              # For importing custom modules (from parent dirs)
sys.path.append('..')
from tools import serialTools, captureTools, eval

##### Global Vars

In [ ]:
datasetsPath = '../datasets/har/'

In [ ]:
xtrain =    pd.DataFrame()
xtest =     pd.DataFrame()
ytrain =    pd.DataFrame()
ytest =     pd.DataFrame()
bestIter = 0
initialAcc = 0

leGender = LabelEncoder()       # Female, Male
leHML = LabelEncoder()          # High, Moderate, Low
leChestPain = LabelEncoder()    # Non-anginal, Asymptomatic, Typical, Atypical
leThalassemia = LabelEncoder()  # Normal, Fixed Defect, Reversible Defect
leECG = LabelEncoder()          # Normal, ST-T abnormality, Left ventricular hypertrophy

##### Func: Import Data

In [ ]:
def importData():
    print(f'Importing data...')

    global xtrain, xtest, ytrain, ytest

    # Load dataset
    data = pd.read_csv(datasetsPath + 'heart_attack_risk_dataset.csv')

    # Encode Categorical Data
    data['Gender'] = leGender.fit_transform(data['Gender'])
    data['Physical_Activity_Level'] = leHML.fit_transform(data['Physical_Activity_Level'])
    data['Stress_Level'] = leHML.fit_transform(data['Stress_Level'])
    data['Heart_Attack_Risk'] = leHML.fit_transform(data['Heart_Attack_Risk'])
    data['Chest_Pain_Type'] = leChestPain.fit_transform(data['Chest_Pain_Type'])
    data['Thalassemia'] = leThalassemia.fit_transform(data['Thalassemia'])
    data['ECG_Results'] = leECG.fit_transform(data['ECG_Results'])

    # Splitting data into features and labels
    xdata = data.iloc[:,:19]
    ydata = data.iloc[:,19:]

    # Splitting data into training and testing
    xtrain, xtest, ytrain, ytest = train_test_split(
        xdata, 
        ydata, 
        test_size=0.2,
        random_state=0
    )
    print(f'Data imported!')

##### Func: Train Model

In [ ]:
def trainModel(model: XGBClassifier, feats: pd.DataFrame, labels: pd.DataFrame, setBestIter: bool = False, evalset: list = None,
               maxDepth: int = 6, gamma: float = 0.0, min_child_weight: float = 1.0, verbose: bool = False):
    
    global bestIter
    
    model.set_params(               # Setting params that do not vary across both options
        objective='multi:softmax',
        num_class=3,
        learning_rate=0.1,
        # subsample=0.9             # default: 1 | [0; 1]
        max_depth=maxDepth,
        # Parameters for Pruning
        gamma=gamma,                    # default: 0 | [0; inf] | alias: min_split_loss
        min_child_weight=min_child_weight,         # default: 1 | [0; inf]   
    )

    if setBestIter == True:
        if verbose == True: print(f'\tTraining model (to determine best iteration)...')
        
        model.set_params(            
            n_estimators=10000,
            early_stopping_rounds=100
        )
        model.fit(
            feats, labels,
            eval_set = evalset,
            verbose = False
        )
        bestIter = model.best_iteration

        if verbose == True: print(f'\tBest iteration: {bestIter}')
    else:
        if verbose == True: print(f'\tTraining model...')

        if bestIter == 0 and verbose == True: print('BestIter = 0 -> Something is wrong!')
        
        model.set_params(
            n_estimators=bestIter,
            early_stopping_rounds=None,
        )
        model.fit(feats,labels)

        if verbose == True: print(f'\tModel trained!')

    

##### Func: Train Quicksave

In [ ]:
def trainQuicksave(maxDepth: int = 6):
    print("Training Quicksave...")
    model = XGBClassifier()
    evalset = [(xtrain,ytrain),(xtest,ytest)]
    trainModel(model, xtrain, ytrain, True, evalset, maxDepth)
    trainModel(model, xtrain, ytrain, maxDepth=maxDepth)
    model.save_model("quicksave.json")
    print("Quicksave model trained and saved as quicksave.json!")

##### Func: Get Amount of Trees

In [ ]:
def getNumTrees(model):
    dump_list = model.get_booster().get_dump()
    num_trees = len(dump_list)
    return num_trees

##### Func: Get Amount of Splits

In [ ]:
def getNumSplits(model):
    trees_strings = model.get_booster().get_dump(dump_format='text')
    total_splits = 0
    for tree_string in trees_strings:
        n_nodes = len(tree_string.split('\n')) - 1
        n_leaves = tree_string.count('leaf')
        total_splits += n_nodes - n_leaves
    return total_splits

##### Func: Plots

In [ ]:
def plot3Subs():    # # Anzeige mit 3 Subplots übereinander

    df = pd.read_csv('har_gamma_metrics.csv')

    fig, axs = pyplot.subplots(3, 1, figsize=(8,12))

    axs[0].plot(df['Gamma'], df['Number of trees'], marker='o', linestyle='-', color='b')
    axs[0].set_title('Number of trees VS Gamma')
    axs[0].set_xlabel('Gamma')
    axs[0].set_ylabel('Number of trees')

    axs[1].plot(df['Gamma'], df['Number of splits'], marker='o', linestyle='-', color='r')
    axs[1].set_title('Number of splits VS Gamma')
    axs[1].set_xlabel('Gamma')
    axs[1].set_ylabel('Number of splits')

    axs[2].plot(df['Gamma'], df['Accuracy'], marker='o', linestyle='-', color='g')
    axs[2].set_title('Accuracy VS Gamma')
    axs[2].set_xlabel('Gamma')
    axs[2].set_ylabel('Accuracy')

    pyplot.tight_layout()
    pyplot.show()

def plot2subs_gamma():    # Kombinierte Anzeige mit 2 Subplots

    df = pd.read_csv('har_gamma_metrics.csv')

    fig, axs = pyplot.subplots(2, 1, figsize=(10, 10))

    ax1 = axs[0]
    ax1.set_xlabel('Gamma')
    ax1.set_ylabel('Number of Trees', color='b')
    # ax1.plot(df['Gamma'], df['Number of trees'], marker='o', linestyle='-', color='b', label='Number of Trees')
    ax1.plot(df['Gamma'], df['Number of trees'], linestyle='-', color='b', label='Number of Trees')
    ax1.tick_params(axis='y', labelcolor='b')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Number of Splits', color='g')
    # ax2.plot(df['Gamma'], df['Number of splits'], marker='s', linestyle='-', color='g', label='Number of Splits')
    ax2.plot(df['Gamma'], df['Number of splits'],  linestyle='-', color='g', label='Number of Splits')
    ax2.tick_params(axis='y', labelcolor='g')

    # axs[1].plot(df['Gamma'], df['Accuracy'], marker='^', linestyle='-', color='r', label='Accuracy Score')
    axs[1].plot(df['Gamma'], df['Accuracy'], linestyle='-', color='r', label='Accuracy Score')
    axs[1].set_title('Accuracy Score vs Gamma')
    axs[1].set_xlabel('Gamma')
    axs[1].set_ylabel('Accuracy Score')

    pyplot.tight_layout()
    pyplot.show()
    
def plot3sharedy(): # 3 Graphen | 1 Plot | geteilte y-Achsen

    df = pd.read_csv('har_gamma_metrics.csv')

    pyplot.figure(figsize=(10, 6))
    pyplot.plot(df['Gamma'], df['Number of trees'], marker='o', linestyle='-', color='b', label='Number of trees')
    pyplot.plot(df['Gamma'], df['Number of splits'], marker='o', linestyle='-', color='r', label='Number of splits')
    pyplot.plot(df['Gamma'], df['Accuracy'], marker='o', linestyle='-', color='g', label='Accuracy')
    pyplot.xlabel('Gamma')
    pyplot.ylabel('Metrics')
    pyplot.title('Metrics VS Gamma')
    pyplot.legend()
    # pyplot.tight_layout()
    pyplot.show()

def plot3disty():   # 3 Graphen | 1 Plot | unterschiedliche y-Achsen

    df = pd.read_csv('har_gamma_metrics.csv')

    fig, ax1 = pyplot.subplots(figsize=(10, 6))

    ax1.set_xlabel('Gamma')
    ax1.set_ylabel('Number of Trees', color='b')
    ax1.plot(df['Gamma'], df['Number of trees'], marker='o', linestyle='-', color='b', label='Number of Trees')
    ax1.tick_params(axis='y', labelcolor='b')

    ax2 = ax1.twinx()  # Erzeugt eine zweite y-Achse
    ax2.set_ylabel('Number of Splits', color='g')
    ax2.plot(df['Gamma'], df['Number of splits'], marker='s', linestyle='-', color='g', label='Number of Splits')
    ax2.tick_params(axis='y', labelcolor='g')

    ax3 = ax1.twinx()  # Erzeugt eine dritte y-Achse
    ax3.spines['right'].set_position(('outward', 60))  # Verschiebt die dritte Achse weiter rechts
    ax3.set_ylabel('Accuracy Score', color='r')
    ax3.plot(df['Gamma'], df['Accuracy'], marker='^', linestyle='-', color='r', label='Accuracy Score')
    ax3.tick_params(axis='y', labelcolor='r')

    pyplot.title('Comparison of Metrics vs Gamma with Multiple Y Axes')

    fig.tight_layout()
    pyplot.show()

def plot2subs_mcws():    # Kombinierte Anzeige mit 2 Subplots - Min Child Weight

    df = pd.read_csv('har_mcws_metrics.csv')

    fig, axs = pyplot.subplots(2, 1, figsize=(10, 10))

    ax1 = axs[0]
    ax1.set_xlabel('Min Child Weight')
    ax1.set_ylabel('Number of Trees', color='b')
    # ax1.plot(df['Min Child Weight'], df['Number of trees'], marker='o', linestyle='-', color='b', label='Number of Trees')
    ax1.plot(df['Min Child Weight'], df['Number of trees'], linestyle='-', color='b', label='Number of Trees')
    ax1.tick_params(axis='y', labelcolor='b')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Number of Splits', color='g')
    # ax2.plot(df['Min Child Weight'], df['Number of splits'], marker='s', linestyle='-', color='g', label='Number of Splits')
    ax2.plot(df['Min Child Weight'], df['Number of splits'],  linestyle='-', color='g', label='Number of Splits')
    ax2.tick_params(axis='y', labelcolor='g')

    # axs[1].plot(df['Min Child Weight'], df['Accuracy'], marker='^', linestyle='-', color='r', label='Accuracy Score')
    axs[1].plot(df['Min Child Weight'], df['Accuracy'], linestyle='-', color='r', label='Accuracy Score')
    axs[1].set_title('Accuracy Score vs Min Child Weight')
    axs[1].set_xlabel('Min Child Weight')
    axs[1].set_ylabel('Accuracy Score')

    pyplot.tight_layout()
    pyplot.show()


##### har_gamma_metrics.csv - Datenerhebung

In [ ]:
# # Main for Pruning
# importData()
# gammas = []
# numTrees = []
# numSplits = []
# accuracies = []

# for i in range(0,70):
#     model = XGBClassifier(random_state=0)
#     trainModel(model, xtrain, ytrain,True, evalset=[(xtrain,ytrain),(xtest,ytest)],gamma=(i/10))
#     trainModel(model, xtrain, ytrain,gamma=(i/10))
#     print(f'Gamma: {i/10}\t|\tNumber of trees: {getNumTrees(model)}\t|\tNumber of splits: {getNumSplits(model)}\t|\tAccuracy Score: {accuracy_score(ytest, model.predict(xtest))}')
#     gammas.append(i/10)
#     numTrees.append(getNumTrees(model))
#     numSplits.append(getNumSplits(model))
#     accuracies.append(accuracy_score(ytest, model.predict(xtest)))

# model = XGBClassifier(random_state=0)
# trainModel(model, xtrain, ytrain,True, evalset=[(xtrain,ytrain),(xtest,ytest)],gamma=7)
# trainModel(model, xtrain, ytrain,gamma=7)
# gammas.append(7.0)
# numTrees.append(getNumTrees(model))
# numSplits.append(getNumSplits(model))
# accuracies.append(accuracy_score(ytest, model.predict(xtest)))

# data = {
#     'Gamma': gammas,
#     'Number of trees': numTrees,
#     'Number of splits': numSplits,
#     'Accuracy': accuracies
# }
# df = pd.DataFrame(data)
# df.to_csv('har_gamma_metrics.csv', index=False)

##### har_mcws_metrics.csv - Datenerhebung

In [ ]:
# importData()
# model = XGBClassifier()

# mcws = []
# numTrees = []
# numSplits = []
# accuracies = []

# for i in range(0,18000,200):
#     trainModel(model, xtrain, ytrain, True, [(xtrain,ytrain),(xtest,ytest)], min_child_weight=i)
#     trainModel(model, xtrain, ytrain, min_child_weight=i)
#     print(f'min_child_weight: {i}\t|\tNumber of trees: {getNumTrees(model)}\t|\tNumber of splits: {getNumSplits(model)}\t|\tAccuracy Score: {accuracy_score(ytest, model.predict(xtest))}')
#     mcws.append(i)
#     numTrees.append(getNumTrees(model))
#     numSplits.append(getNumSplits(model))
#     accuracies.append(accuracy_score(ytest, model.predict(xtest)))

# # Ich habe es schon wieder vergessen!
# trainModel(model, xtrain, ytrain, True, [(xtrain,ytrain),(xtest,ytest)], min_child_weight=18000)
# trainModel(model, xtrain, ytrain, min_child_weight=18000)
# print(f'min_child_weight: {i}\t|\tNumber of trees: {getNumTrees(model)}\t|\tNumber of splits: {getNumSplits(model)}\t|\tAccuracy Score: {accuracy_score(ytest, model.predict(xtest))}')
# mcws.append(i)
# numTrees.append(getNumTrees(model))
# numSplits.append(getNumSplits(model))
# accuracies.append(accuracy_score(ytest, model.predict(xtest)))

# data = {
#     'min_child_weight': mcws,
#     'Number of trees': numTrees,
#     'Number of splits': numSplits,
#     'Accuracy': accuracies
# }
# df = pd.DataFrame(data)
# df.to_csv('har_mcw_metrics.csv', index=False)
    

##### Main

In [ ]:
# importData()
# trainQuicksave(6)
# model = XGBClassifier()
# model.load_model("quicksave.json")
# print(f'Number of trees: {getNumTrees(model)}')
# print(f'Number of splits: {getNumSplits(model)}')
# print(accuracy_score(ytest, model.predict(xtest)))